# JSON → Parquet Pipeline (Pydantic)

## Goal

This notebook implements a **Python-only** data processing workflow for nested JSON records (e.g. from `synthetic_source.jsonl`) using **Pydantic v2** for schema enforcement and normalization—**no Spark**.

**Pipeline stages:**
1. **Raw JSON** → read JSONL or `.json` files into Python dicts.
2. **Bronze Parquet** → write raw records to Parquet with a `created_date` partition column (Hive-style); preserve nested structure.
3. **Silver Parquet** → parse and cleanse via Pydantic models (normalize missing values, parse dates, clean text, add derived fields), deduplicate by `id`, then write partitioned Parquet.
4. **DQ Report** → JSON report with row counts, parse failures, duplicate counts, missing/null rates; invalid records written to `rejects.jsonl`.

**Why this design:** Build a clear, reference pipeline in Python so that each transformation is explicit and easy to port to Spark later (read → schema → cleansing → dedupe → write).

## 2. Setup

Imports, config, and directory creation.

In [19]:
# Python 3.11. Requirements: pydantic>=2, pyarrow, pandas (pip install pydantic pyarrow pandas)
try:
    import json
    import os
    from datetime import datetime, timezone
    from pathlib import Path
    from typing import Any, Optional

    import pyarrow as pa
    import pyarrow.dataset as ds
    import pandas as pd
    from pydantic import BaseModel, ConfigDict, model_validator, Field

    # ---------------------------------------------------------------------------
    # Config (tune paths and text handling)
    # ---------------------------------------------------------------------------
    INPUT_JSON_DIR = "./data"           # directory containing .jsonl or .json files
    BRONZE_DIR = "./bronze"
    SILVER_DIR = "./silver"
    DQ_REPORT_PATH = "./dq_report.json"
    TRUNCATE_TEXT = True
    MAX_TEXT_LEN = 50_000

    # Create output dirs
    for d in (INPUT_JSON_DIR, BRONZE_DIR, SILVER_DIR):
        os.makedirs(d, exist_ok=True)
    # Rejects file next to DQ report
    DQ_REJECTS_PATH = str(Path(DQ_REPORT_PATH).with_suffix("")) + "_rejects.jsonl"
    print(f"Config: INPUT_JSON_DIR={INPUT_JSON_DIR}, BRONZE={BRONZE_DIR}, SILVER={SILVER_DIR}")
    print(f"DQ report: {DQ_REPORT_PATH}, rejects: {DQ_REJECTS_PATH}")
    print(f"TRUNCATE_TEXT={TRUNCATE_TEXT}, MAX_TEXT_LEN={MAX_TEXT_LEN}")
except Exception as e:
    # Dummy fallback so downstream cells don't crash (e.g. missing pyarrow/pydantic)
    import json
    import os
    from pathlib import Path
    INPUT_JSON_DIR = "./data"
    BRONZE_DIR = "./bronze"
    SILVER_DIR = "./silver"
    DQ_REPORT_PATH = "./dq_report.json"
    DQ_REJECTS_PATH = "dq_report_rejects.jsonl"
    TRUNCATE_TEXT = True
    MAX_TEXT_LEN = 50_000
    for d in (INPUT_JSON_DIR, BRONZE_DIR, SILVER_DIR):
        os.makedirs(d, exist_ok=True)
    print(f"[Dummy config] Setup failed: {e}. Using paths only; run 'pip install pydantic pyarrow pandas' and re-run.")

Config: INPUT_JSON_DIR=./data, BRONZE=./bronze, SILVER=./silver
DQ report: ./dq_report.json, rejects: dq_report_rejects.jsonl
TRUNCATE_TEXT=True, MAX_TEXT_LEN=50000


## 3. Pydantic models (explicit schema)

All input fields are optional; we normalize missing nested objects to `{}` and missing arrays to `[]`. We allow extra keys so future fields do not break parsing. Type enforcement and derived fields (e.g. `created_at_ts`, `workload_key`, `fault_key`, text `*_len` fields) are implemented via validators/serializers.

In [20]:
# --- Helpers used in validators (text cleanup) ---
def _clean_text(raw: Any, truncate: bool, max_len: int) -> tuple[str, int]:
    """Return (cleaned_text, length_before_truncation). Length is of cleaned text."""
    if raw is None:
        return "", 0
    s = str(raw).replace("\x00", "").replace("\r\n", "\n").replace("\r", "\n")
    length = len(s)
    if truncate and length > max_len:
        s = s[:max_len]
    return s, length

# --- Sub-models: allow extra keys, default missing to empty ---
class Context(BaseModel):
    model_config = ConfigDict(extra="allow")
    cluster_id: Optional[str] = None
    namespace: Optional[str] = None
    workload_kind: Optional[str] = None
    workload_name: Optional[str] = None
    container_name: Optional[str] = None
    image: Optional[str] = None
    gitops_enabled: Optional[bool] = None
    gitops_tool: Optional[str] = None
    repo_path: Optional[str] = None
    workload_key: Optional[str] = None  # derived: namespace/kind/name

    @model_validator(mode="after")
    def set_workload_key(self):
        parts = [self.namespace or "", self.workload_kind or "", self.workload_name or ""]
        self.workload_key = "/".join(p for p in parts if p) or None
        return self


class World(BaseModel):
    model_config = ConfigDict(extra="allow")
    nodes: list[Any] = Field(default_factory=list)
    workload_spec: dict[str, Any] = Field(default_factory=dict)
    inventory: dict[str, Any] = Field(default_factory=dict)
    gitops: dict[str, Any] = Field(default_factory=dict)


class Fault(BaseModel):
    model_config = ConfigDict(extra="allow")
    scenario_id: Optional[str] = None
    variant: Optional[str] = None
    fault_params: dict[str, Any] = Field(default_factory=dict)
    fault_key: Optional[str] = None  # derived: scenario_id:variant

    @model_validator(mode="after")
    def set_fault_key(self):
        key = f"{self.scenario_id or ''}:{self.variant or ''}".strip(":")
        self.fault_key = key if key else None
        return self


# Observations: text fields cleaned; companion *_len fields store length before truncation
class Observations(BaseModel):
    model_config = ConfigDict(extra="allow")
    kubectl_get_pods: Optional[str] = None
    kubectl_describe_pod: Optional[str] = None
    kubectl_get_events: Optional[str] = None
    container_logs: Optional[str] = None
    gitops_status: Optional[str] = None
    metrics_snapshot: dict[str, Any] = Field(default_factory=dict)
    # Lengths (cleaned, before truncation) - set in validator
    kubectl_get_pods_len: Optional[int] = None
    kubectl_describe_pod_len: Optional[int] = None
    kubectl_get_events_len: Optional[int] = None
    container_logs_len: Optional[int] = None

    @model_validator(mode="after")
    def clean_text_and_set_lens(self):
        truncate = TRUNCATE_TEXT
        max_len = MAX_TEXT_LEN
        for name, len_name in [
            ("kubectl_get_pods", "kubectl_get_pods_len"),
            ("kubectl_describe_pod", "kubectl_describe_pod_len"),
            ("kubectl_get_events", "kubectl_get_events_len"),
            ("container_logs", "container_logs_len"),
        ]:
            raw = getattr(self, name)
            text, length = _clean_text(raw, truncate, max_len)
            setattr(self, name, text)
            setattr(self, len_name, length)
        if self.gitops_status is not None:
            self.gitops_status, _ = _clean_text(self.gitops_status, truncate, max_len)
        return self


class Remediation(BaseModel):
    model_config = ConfigDict(extra="allow")
    category: Optional[str] = None
    diagnosis: Optional[str] = None
    fix_plan: list[Any] = Field(default_factory=list)
    actions_structured: list[Any] = Field(default_factory=list)
    verification: list[Any] = Field(default_factory=list)
    rollback: list[Any] = Field(default_factory=list)


class Meta(BaseModel):
    model_config = ConfigDict(extra="allow")
    created_at: Optional[str] = None
    difficulty: Optional[str] = None
    noise_level: Optional[float] = None
    created_at_ts: Optional[str] = None   # ISO datetime string (UTC) after parsing
    created_date: Optional[str] = None    # YYYY-MM-DD or "unknown"

    @model_validator(mode="after")
    def parse_created_at(self):
        raw = self.created_at
        if not raw or not str(raw).strip():
            self.created_at_ts = None
            self.created_date = "unknown"
            return self
        try:
            # Try ISO format first, then common variants
            s = str(raw).strip()
            for fmt in ("%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
                try:
                    dt = datetime.strptime(s.replace("Z", "+00:00"), fmt.replace("Z", "%z") if "Z" in fmt else fmt)
                    if dt.tzinfo is None:
                        dt = dt.replace(tzinfo=timezone.utc)
                    else:
                        dt = dt.astimezone(timezone.utc)
                    self.created_at_ts = dt.isoformat()
                    self.created_date = dt.strftime("%Y-%m-%d")
                    return self
                except ValueError:
                    continue
            # last resort: fromisoformat
            dt = datetime.fromisoformat(s)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            else:
                dt = dt.astimezone(timezone.utc)
            self.created_at_ts = dt.isoformat()
            self.created_date = dt.strftime("%Y-%m-%d")
        except Exception:
            self.created_at_ts = None
            self.created_date = "unknown"
        return self


class Record(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: Optional[str] = None
    context: Optional[Context] = None
    world: Optional[World] = None
    fault: Optional[Fault] = None
    observations: Optional[Observations] = None
    remediation: Optional[Remediation] = None
    meta: Optional[Meta] = None

    @model_validator(mode="before")
    @classmethod
    def normalize_nested(cls, data: Any):
        if not isinstance(data, dict):
            return data
        # Missing nested -> empty dict so Pydantic builds sub-models with defaults
        for key in ("context", "world", "fault", "observations", "remediation", "meta"):
            if key not in data or data[key] is None:
                data[key] = {}
        return data

print("Pydantic models defined (Context, World, Fault, Observations, Remediation, Meta, Record).")

Pydantic models defined (Context, World, Fault, Observations, Remediation, Meta, Record).


## 4. Normalization rules (in models)

- **created_at:** Parsed in `Meta` into timezone-aware UTC → `created_at_ts` (ISO str) and `created_date` (YYYY-MM-DD); on failure → `created_date = "unknown"`, `created_at_ts = None`. Parse failure count is collected in the DQ report.
- **Text cleanup** (in `Observations`): `kubectl_get_pods`, `kubectl_describe_pod`, `kubectl_get_events`, `container_logs`: None→`""`, strip null bytes, normalize newlines; if `TRUNCATE_TEXT` then truncate to `MAX_TEXT_LEN`. Companion fields `*_len` store the length of the cleaned text *before* truncation.
- **Derived:** `context.workload_key = "{namespace}/{workload_kind}/{workload_name}"` (skip missing); `fault.fault_key = "{scenario_id}:{variant}"`.

## 5. Read raw JSON

Load from `INPUT_JSON_DIR`: support both `.jsonl` (one JSON object per line) and single `.json` (array of objects). Result is a list of Python dicts.

*Performance:* For very large JSONL, read line-by-line in a loop and process in batches (e.g. parse with Pydantic per batch, write Parquet per batch) to avoid loading everything into memory; the logic below keeps the pipeline straightforward.

In [21]:
def load_raw_json(input_dir: str) -> list[dict]:
    input_path = Path(input_dir)
    if not input_path.exists():
        return []
    records: list[dict] = []
    for p in sorted(input_path.iterdir()):
        if p.suffix.lower() == ".jsonl":
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    records.append(json.loads(line))
        elif p.suffix.lower() == ".json":
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    records.extend(data)
                else:
                    records.append(data)
    return records

raw_records = load_raw_json(INPUT_JSON_DIR)
print(f"Loaded {len(raw_records)} records from {INPUT_JSON_DIR}")
if raw_records:
    print("Example record (first, keys only):", list(raw_records[0].keys()))
    print("Sample (id, context.namespace, fault.scenario_id):", {
        "id": raw_records[0].get("id"),
        "namespace": (raw_records[0].get("context") or {}).get("namespace"),
        "scenario_id": (raw_records[0].get("fault") or {}).get("scenario_id"),
    })

Loaded 1 records from ./data
Example record (first, keys only): ['id', 'context', 'world', 'fault', 'observations', 'remediation', 'meta']
Sample (id, context.namespace, fault.scenario_id): {'id': 'test-1', 'namespace': 'ns1', 'scenario_id': 'x'}


## 6. Bronze write (raw → Parquet)

Preserve raw dicts; add a `created_date` partition column (from `meta.created_at`; simple parse; `"unknown"` if missing/unparseable). Write with Hive-style partitioning using `pyarrow.dataset.write_dataset`.

In [22]:
def _bronze_created_date(rec: dict) -> str:
    """Derive partition column from meta.created_at (simple parse)."""
    raw = (rec.get("meta") or {}).get("created_at")
    if not raw:
        return "unknown"
    s = str(raw).strip()
    try:
        # Use first 10 chars for date-only, or 19 for datetime
        for part, fmt in [(s[:10], "%Y-%m-%d"), (s[:19].replace("Z", ""), "%Y-%m-%dT%H:%M:%S"), (s[:19], "%Y-%m-%d %H:%M:%S")]:
            try:
                dt = datetime.strptime(part, fmt)
                return dt.strftime("%Y-%m-%d")
            except ValueError:
                continue
        return "unknown"
    except Exception:
        return "unknown"

# Add partition column to each raw record (copy so we don't mutate originals)
bronze_records = []
for r in raw_records:
    row = {**r, "created_date": _bronze_created_date(r)}
    bronze_records.append(row)

# Parquet cannot write structs with zero fields; fill empty dicts with a dummy key
def _no_empty_structs(obj):
    if isinstance(obj, dict):
        if len(obj) == 0:
            return {"_": None}
        return {k: _no_empty_structs(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_no_empty_structs(x) for x in obj]
    return obj

if bronze_records:
    safe_bronze = [_no_empty_structs(row) for row in bronze_records]
    bronze_table = pa.Table.from_pylist(safe_bronze)
    ds.write_dataset(
        bronze_table,
        base_dir=BRONZE_DIR,
        format="parquet",
        partitioning=ds.partitioning(pa.schema([("created_date", pa.string())]), flavor="hive"),
        existing_data_behavior="overwrite_or_ignore",
    )
partitions = list(Path(BRONZE_DIR).rglob("*.parquet"))
print(f"Bronze: wrote {len(bronze_records)} rows to {BRONZE_DIR}; partition files: {len(partitions)}")

Bronze: wrote 1 rows to ./bronze; partition files: 1


## 7. Silver processing (parse + cleanse via Pydantic)

Parse each raw dict into `Record`; collect validation errors into a rejects list (with error message). Convert validated models to dicts for Parquet. Deduplicate by `id`: keep latest by `meta.created_at_ts` (None sorts last; ties keep first). Report duplicates removed.

In [23]:
validated: list[Record] = []
rejects: list[dict] = []  # [{ "raw_record": {...}, "error": "..." }]

for raw in raw_records:
    try:
        r = Record.model_validate(raw)
        validated.append(r)
    except Exception as e:
        rejects.append({"raw_record": raw, "error": str(e)})

# Convert to dicts (for Parquet); include all normalized and derived fields
def record_to_dict(r: Record) -> dict:
    d = r.model_dump(mode="json")
    # Pydantic already put created_at_ts, created_date, workload_key, fault_key, *_len in the model
    return d

silver_dicts = [record_to_dict(r) for r in validated]

# Deduplicate by id: keep latest by meta.created_at_ts (None last); on tie keep first
by_id: dict[str, dict] = {}
for d in silver_dicts:
    rid = d.get("id") or ""
    ts_str = (d.get("meta") or {}).get("created_at_ts")
    incumbent = by_id.get(rid)
    if incumbent is None:
        by_id[rid] = d
        continue
    inc_ts = (incumbent.get("meta") or {}).get("created_at_ts")
    # None sorts last
    if (ts_str or "") > (inc_ts or ""):
        by_id[rid] = d

silver_deduped = list(by_id.values())
duplicates_removed = len(silver_dicts) - len(silver_deduped)
print(f"Valid: {len(validated)}, Rejected: {len(rejects)}, After dedupe: {len(silver_deduped)}, Duplicates removed: {duplicates_removed}")

Valid: 1, Rejected: 0, After dedupe: 1, Duplicates removed: 0


## 8. Silver write

Write silver table to Parquet partitioned by `created_date` (from `meta.created_date`). Add top-level `created_date` column for Hive partitioning.

In [24]:
# Parquet cannot write structs with zero fields; ensure no empty dicts (add dummy field)
def _no_empty_structs(obj):
    if isinstance(obj, dict):
        if len(obj) == 0:
            return {"_": None}
        return {k: _no_empty_structs(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_no_empty_structs(x) for x in obj]
    return obj

# Add partition column for write_dataset
for d in silver_deduped:
    d["created_date"] = (d.get("meta") or {}).get("created_date") or "unknown"

if silver_deduped:
    safe_silver = [_no_empty_structs(d) for d in silver_deduped]
    silver_table = pa.Table.from_pylist(safe_silver)
    ds.write_dataset(
        silver_table,
        base_dir=SILVER_DIR,
        format="parquet",
        partitioning=ds.partitioning(pa.schema([("created_date", pa.string())]), flavor="hive"),
        existing_data_behavior="overwrite_or_ignore",
    )
silver_partitions = list(Path(SILVER_DIR).rglob("*.parquet"))
print(f"Silver: wrote {len(silver_deduped)} rows to {SILVER_DIR}; partition files: {len(silver_partitions)}")

Silver: wrote 1 rows to ./silver; partition files: 1


## 9. Data Quality report

Build a report dict with row counts, parse failures, duplicate count, missing-key counts, null/empty rates for log fields. Write rejects to `*_rejects.jsonl`; save report JSON and display.

In [25]:
# Created_at parse failures: had meta.created_at but created_at_ts is None (or created_date == "unknown")
created_at_parse_failures = 0
for d in silver_deduped:
    meta = d.get("meta") or {}
    if (meta.get("created_at") or "").strip() and (not meta.get("created_at_ts") or meta.get("created_date") == "unknown"):
        created_at_parse_failures += 1

# Missing key counts (on raw_records for id/context/meta; on silver_deduped for consistency)
def _missing(rows: list[dict], path: str) -> int:
    parts = path.split(".")
    n = 0
    for r in rows:
        cur = r
        for p in parts:
            cur = (cur or {}).get(p) if isinstance(cur, dict) else None
            if cur is None and p != parts[-1]:
                n += 1
                break
        else:
            if cur is None or (isinstance(cur, str) and not cur.strip()):
                n += 1
    return n

missing_id = sum(1 for r in raw_records if not (r.get("id") or "").strip())
missing_cluster_id = sum(1 for r in raw_records if not ((r.get("context") or {}).get("cluster_id") or "").strip())
missing_created_at = sum(1 for r in raw_records if not ((r.get("meta") or {}).get("created_at") or "").strip())

# Null/empty rates for log fields (on silver_deduped)
obs_keys = ["kubectl_get_pods", "kubectl_describe_pod", "kubectl_get_events", "container_logs"]
null_empty_logs = {}
for k in obs_keys:
    null_empty_logs[k] = sum(1 for d in silver_deduped if not ((d.get("observations") or {}).get(k) or "").strip())

report = {
    "total_rows_loaded": len(raw_records),
    "total_rows_bronze_written": len(bronze_records),
    "total_rows_valid": len(validated),
    "total_rows_rejected": len(rejects),
    "created_at_parse_failures": created_at_parse_failures,
    "duplicates_removed": duplicates_removed,
    "missing": {
        "id": missing_id,
        "context.cluster_id": missing_cluster_id,
        "meta.created_at": missing_created_at,
    },
    "null_empty_observations": null_empty_logs,
}

with open(DQ_REJECTS_PATH, "w", encoding="utf-8") as f:
    for item in rejects:
        f.write(json.dumps(item, default=str) + "\n")

with open(DQ_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("DQ report:")
print(json.dumps(report, indent=2))
print(f"\nRejects written to {DQ_REJECTS_PATH}")

# Tabular summary (pandas)
summary = pd.DataFrame([{
    "stage": "loaded", "rows": report["total_rows_loaded"],
}, {"stage": "bronze", "rows": report["total_rows_bronze_written"]}, {"stage": "valid", "rows": report["total_rows_valid"]}, {"stage": "rejected", "rows": report["total_rows_rejected"]}, {"stage": "silver (deduped)", "rows": report["total_rows_valid"] - report["duplicates_removed"]}])
print("\nRow counts by stage:")
print(summary.to_string(index=False))

DQ report:
{
  "total_rows_loaded": 1,
  "total_rows_bronze_written": 1,
  "total_rows_valid": 1,
  "total_rows_rejected": 0,
  "created_at_parse_failures": 0,
  "duplicates_removed": 0,
  "missing": {
    "id": 0,
    "context.cluster_id": 0,
    "meta.created_at": 0
  },
  "null_empty_observations": {
    "kubectl_get_pods": 0,
    "kubectl_describe_pod": 1,
    "kubectl_get_events": 1,
    "container_logs": 1
  }
}

Rejects written to dq_report_rejects.jsonl

Row counts by stage:
           stage  rows
          loaded     1
          bronze     1
           valid     1
        rejected     0
silver (deduped)     1


## 10. Porting to Spark later

| Step | Python (this notebook) | Spark mapping |
|------|------------------------|----------------|
| **Read** | `load_raw_json()` → list of dicts | `spark.read.json(path)` or `spark.read.format("json").load()` with multiline / JSONL options |
| **Schema** | Pydantic `Record` + validators | Define `StructType` / `StructField` from the same logical schema; keep same field names and nesting |
| **Cleansing** | Parse with `Record.model_validate()`; normalizers in validators | Apply same rules in Spark: `withColumn` + UDF or Spark SQL (e.g. `regexp_replace`, `when/otherwise`), or use `mapInPandas` with the same Pydantic models per partition |
| **Dedupe** | Dict by `id`, keep max `created_at_ts` | `window_partition_by("id").order_by(col("meta.created_at_ts").desc_null_last()).row_number() == 1` then filter |
| **Write** | `pyarrow.dataset.write_dataset(..., partitioning=created_date)` | `df.write.partitionBy("created_date").parquet(silver_path)` |

The Pydantic layer keeps the rules explicit and testable; Spark re-implements the same logic (or calls into Python via `mapInPandas`) for scale.